# Contact Center Centralized FY2025

This notebook computes Contact Center risk metrics for FY2025.

**Metrics Computed:**
- **1.1** — Total number of unrated or unscored customers
- **1.2** — Tier 1/2 High Risk Customers
- **1.3** — High Risk Customers (excluding Tier 1/2)
- **1.4** — Medium Risk Customers
- **1.5** — Low Risk Customers (including unscored)
- **6.5b** — Total number of customers at end of review period
- **SD2/ABAC PEP** — PEP (Politically Exposed Persons) analysis

**NACO Universe:** Deduplicated by `customr_num` using `MIN(customr_type)` to match the 6.5b Method B count (business standard).

## Setup: JDBC Connections & Credentials

In [ ]:
# ------------------------------------------------------------------
# Cell 1: JDBC connection strings and authentication credentials
# ------------------------------------------------------------------
# Define JDBC URLs for various SQL Server pools used by the pipeline.
# Credentials are fetched from Databricks secrets (Azure AD Service Principal).
# ------------------------------------------------------------------

czJdbcUrl = "jdbc:sqlserver://p3001-eastus2-asql-2.database.windows.net:1433;database=eda-akora2-aaecz-corporatepoolprd;loginTimeout=10;"
srzJdbcURL = "jdbc:sqlserver://p3001-eastus2-asql-3.database.windows.net:1433;database=eda-akora2-aaed1-srzpoolprd;loginTimeout=10"
azJdbcURL = "jdbc:sqlserver://p3006-eastus2-asql-1.database.windows.net:1433;database=eda-akora-aaaz-CAGAML00BI0001ClusterPRD;loginTimeout=10"

izJdbcUrl = "jdbc:sqlserver://p3004-eastus2-asql-32.database.windows.net:1433;database=eda-akora-aaicz-icz00001poolprd;loginTimeout=10"

jdbcUsername = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_AppID")
jdbcPassword = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_PWD")

connectionProperties = {
    "AADSecurePrincipalID" : jdbcUsername,
    "AADSecurePrincipalSecret" : jdbcPassword,
    "driver" : "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "authentication" : "ActiveDirectoryServicePrincipal",
    "fetchsize" : "10"
}

## Build NACO (National Account Centre Operations) Universe

Create the Contact Centre AU (Assessment Unit) table for the reporting date **20251031**.

The universe combines:
- **Personal customers** (customr_type = 0) joined with `cif_personal_fy25`
- **Non-personal customers** (customr_type = 1) joined with `cif_non_personal_fy25`

**Deduplication:** `GROUP BY customr_num` with `MIN(customr_type)` ensures each customer appears exactly once.
For dual-type customers, personal (type=0) takes priority. This matches the 6.5b Method B count.

Filters applied:
- Bank number = 4
- Application ID in ('ACS', 'VSA')
- Effective date on or before 20251031
- Customer status = '00' (active)

In [ ]:
%sql
-- ==================================================================
-- Build the NACO (Contact Centre) Assessment Unit universe.
-- Deduplicates by customr_num using MIN(customr_type) so that each
-- customer appears exactly once. For dual-type customers, personal
-- (type=0) takes priority over non-personal (type=1).
-- This matches the 6.5b Method B count reported to business.
-- ==================================================================
CREATE OR REPLACE TABLE rafy2025_centralized.contact_centre_au_20251031
USING DELTA
AS
SELECT customr_num, MIN(customr_type) as customr_type
FROM (
  SELECT DISTINCT acc.customr_num, acc.customr_type
  FROM ra_fy_2025.cif_accounts_fy25 acc
  JOIN ra_fy_2025.cif_personal_fy25 pers ON acc.customr_num = pers.customr_num AND acc.customr_bank_num = pers.customr_bank_num AND acc.customr_type = pers.customr_type
  WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 0
  AND acc.aplictn_id in ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND pers.customr_status = '00'
  UNION
  SELECT DISTINCT acc.customr_num, acc.customr_type
  FROM ra_fy_2025.cif_accounts_fy25 acc
  JOIN ra_fy_2025.cif_non_personal_fy25 npers ON acc.customr_num = npers.customr_num AND acc.customr_bank_num = npers.customr_bank_num AND acc.customr_type = npers.customr_type
  WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 1
  AND acc.aplictn_id in ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND npers.customr_status = '00'
)
GROUP BY customr_num

## Data Preparation: Load NACO Universe & Build CIF Keys

In [ ]:
# ------------------------------------------------------------------
# Cell 15: Import PySpark SQL functions
# ------------------------------------------------------------------
from pyspark.sql.functions import *

In [ ]:
# ------------------------------------------------------------------
# Cell 16: Load the NACO universe table created in the SQL cell above
# ------------------------------------------------------------------
naco = spark.table("rafy2025_centralized.contact_centre_au_20251031")

In [ ]:
# ------------------------------------------------------------------
# Cell 17: Derive standardized CIF keys from the NACO universe
#   - cust_no  : 9-digit zero-padded customer number
#   - cust_type: 'N' for non-personal (customr_type=1), 'P' for personal
# ------------------------------------------------------------------
cif = naco.withColumn('cust_no', lpad(col('customr_num'), 9, '0')).withColumn('cust_type', when(col('customr_type') == '1', 'N').otherwise('P'))

In [ ]:
# ------------------------------------------------------------------
# Cell 18: Read the latest customer record from CAEDW via JDBC
#   - Uses row_number() to get the most recent record per cust_id
#   - Pulls cust_id, cust_no, cust_type_mn
# ------------------------------------------------------------------
cust = '''(select cust_id, cust_no, cust_type_mn from (select *, row_number() over (partition by cust_id order by to_dt desc) as row_num from caedw.cust) c where c.row_num = 1)t'''
df_cust = spark.read.jdbc(url = czJdbcUrl, table = cust, properties = connectionProperties).cache()

In [ ]:
# ------------------------------------------------------------------
# Cell 19: Join CIF keys with CAEDW customer data to enrich the
#          NACO universe with cust_id and cust_type_mn.
#   - Left join ensures all NACO records are retained
#   - Drop duplicate cust_no column from df_cust after join
# ------------------------------------------------------------------
niu_naco = cif.join(df_cust, (cif.cust_no == df_cust.cust_no) & (cif.cust_type == df_cust.cust_type_mn), how = 'left').drop(df_cust.cust_no)

## Metric 1.1 — Total Number of Unrated or Unscored Customers

Identifies NACO customers that do **not** appear in the scored/rated customer table.
Uses a `left_anti` join to find unscored CIF customers.

In [ ]:
# ------------------------------------------------------------------
# Cell 21: Metric 1.1 — Unrated / Unscored Customers
#   1. Load the scored customer table (CRR_Scorable_Cust_CA)
#   2. Filter to CIF-prefixed entity IDs only
#   3. Derive cust_no and cust_type_mn from v_entity_id
#   4. Left-anti join with NACO universe to find unscored customers
#   5. Print the count
# ------------------------------------------------------------------

# The 1.1 logic by filtering the scored table by CIF prefix in the v_entity_id
scored_src = spark.table("rafy2025_centralized.CRR_Scorable_Cust_CA")
scored_keys = scored_src.filter(col('v_entity_id').startswith('CIF'))
scored_keys = scored_keys.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))

print("scored CIF distinct customers (cust_no) =", scored_keys.count())

metric_1_1 = (
    niu_naco
    .join(scored_keys, on = ["cust_no", "cust_type_mn"], how="left_anti")
    .dropDuplicates()
    )

metric_1_1_count = metric_1_1.count()

print("Metric 1.1 (CIF unscored found in AU) =", metric_1_1_count)

## Metric 1.2 — Tier 1/2 High Risk Customers

Counts distinct customers in the NACO universe whose risk rating
is **Tier 1** or **Tier 2** (the highest risk tiers).

In [ ]:
# ------------------------------------------------------------------
# Cell 28: Load rated customers and prepare CIF keys for Tier 1/2
# ------------------------------------------------------------------

# Tier 1/2 High risk customers
hrc1 = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
hrc_cif1 = hrc1.filter(col('v_entity_id').startswith('CIF'))
hrc_cif1 = hrc_cif1.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))
display(hrc_cif1)

In [ ]:
# ------------------------------------------------------------------
# Cell 29: Metric 1.2 — Count distinct Tier 1/2 customers in NACO
# ------------------------------------------------------------------
contact_center_hrc1 = hrc_cif1.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner').filter(col('risk_rating').isin("Tier 1", "Tier 2"))
contact_center_hrc1.agg(countDistinct('cust_intrl_id').alias('1.2')).display()

## Metric 1.3 — High Risk Customers (Excluding Tier 1/2)

Counts distinct customers in the NACO universe with a **High** risk
rating (but not Tier 1 or Tier 2).

In [ ]:
# ------------------------------------------------------------------
# Cell 31: Load rated customers and prepare CIF keys for High risk
# ------------------------------------------------------------------

# 1.3 High Risk Customers (exclude tier 1/2)
hrc = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
hrc_cif = hrc.filter(col('v_entity_id').startswith('CIF'))
hrc_cif = hrc_cif.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))
display(hrc_cif)

In [ ]:
# ------------------------------------------------------------------
# Cell 32: Metric 1.3 — Count distinct High risk customers in NACO
# ------------------------------------------------------------------
contact_center_hrc = hrc_cif.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner').filter(col('risk_rating') == "High")
contact_center_hrc.agg(countDistinct('cust_intrl_id').alias('1.3')).display()

## Metric 1.4 — Medium Risk Customers

Counts distinct customers in the NACO universe with a **Medium** risk rating.

In [ ]:
# ------------------------------------------------------------------
# Cell 34: Load rated customers and prepare CIF keys for Medium risk
# ------------------------------------------------------------------

# 1.4 Medium Risk Customers
hrc2 = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
hrc_cif2 = hrc2.filter(col('v_entity_id').startswith('CIF'))
hrc_cif2 = hrc_cif2.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))
display(hrc_cif2)

In [ ]:
# ------------------------------------------------------------------
# Cell 35: Metric 1.4 — Count distinct Medium risk customers in NACO
# ------------------------------------------------------------------
contact_center_hrc2 = hrc_cif2.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner').filter(col('risk_rating') == "Medium")
contact_center_hrc2.agg(countDistinct('cust_intrl_id').alias('1.4')).display()

## Metric 1.5 — Low Risk Customers (Including Unscored)

Combines two populations:
1. **Low-rated** customers from the rated table (risk_rating = 'Low')
2. **Unscored/unrated** customers from the unrated table

These are unioned together and then counted.

In [ ]:
# ------------------------------------------------------------------
# Cell 37: Load both unrated and rated customer tables, prepare CIF keys
# ------------------------------------------------------------------

# 1.5 Low Risk Customers
hrc3_unscored = spark.table('rafy2025_centralized.customer_scorable_unrated_ca')
hrc_cif3_unscored = hrc3_unscored.filter(col('v_entity_id').startswith('CIF'))
hrc_cif3_unscored = hrc_cif3_unscored.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))
hrc3 = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
hrc_cif3 = hrc3.filter(col('v_entity_id').startswith('CIF'))
hrc_cif3 = hrc_cif3.withColumn('cust_no', substring(col('v_entity_id'), -9, 9)).withColumn('cust_type_mn', when(substring(col("v_entity_id"), 8, 1) == "1", "N").otherwise("P"))
display(hrc_cif3_unscored)

In [ ]:
# ------------------------------------------------------------------
# Cell 38: Metric 1.5 — Low risk + unscored customers in NACO
#   - Filter rated customers to 'Low' risk rating
#   - Join unscored customers with NACO universe
#   - Union both sets (allowMissingColumns handles schema differences)
#   - Count distinct cust_intrl_id
# ------------------------------------------------------------------
contact_center_hrc3_low = hrc_cif3.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner').filter(col('risk_rating') == 'Low')
contact_center_hrc3_unscored = hrc_cif3_unscored.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner')
contact_center_hrc3 = contact_center_hrc3_low.unionByName(contact_center_hrc3_unscored, allowMissingColumns=True)
contact_center_hrc3.agg(countDistinct('cust_intrl_id').alias('1.5')).display()

## Metric 6.5b — Total Number of Customers at End of Review Period

Two approaches exist for counting the AU universe:
- **Method A** (larger): `COUNT(DISTINCT customr_num, customr_type)` — counts each customer-type combination separately
- **Method B** (smaller, used by business): `COUNT(DISTINCT customr_num)` — deduplicates across personal/non-personal

The business reports use **Method B**.

In [ ]:
%sql
-- ============================================================
-- 6.5b Method A: COUNT by (customr_num, customr_type) — LARGER
-- This is the OLD universe before NACO dedup. NOT used for reporting.
-- Result: ~16,851,015
-- ============================================================
select count(*) as 6_5b from
(
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_personal_fy25 pers ON acc.customr_num = pers.customr_num AND acc.customr_bank_num = pers.customr_bank_num AND acc.customr_type = pers.customr_type
WHERE acc.customr_bank_num = 4
AND acc.customr_type = 0
AND acc.aplictn_id in ('ACS', 'VSA')
AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
AND pers.customr_status = '00'
UNION
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_non_personal_fy25 npers ON acc.customr_num = npers.customr_num AND acc.customr_bank_num = npers.customr_bank_num AND acc.customr_type = npers.customr_type
WHERE acc.customr_bank_num = 4
AND acc.customr_type = 1
AND acc.aplictn_id in ('ACS', 'VSA')
AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
AND npers.customr_status = '00'
)

In [ ]:
%sql
-- ============================================================
-- 6.5b Method B: COUNT by customr_num ONLY — SMALLER
-- This is what is reported to business.
-- Result: ~16,784,275
-- ============================================================
SELECT COUNT(*) as SD_1 FROM
(
SELECT DISTINCT acc.customr_num
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_personal_fy25 pers ON acc.customr_num = pers.customr_num AND acc.customr_bank_num = pers.customr_bank_num AND acc.customr_type = pers.customr_type
WHERE acc.customr_bank_num = 4
AND acc.customr_type = 0
AND acc.aplictn_id in ('ACS', 'VSA')
AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
AND pers.customr_status = '00'
UNION
SELECT DISTINCT acc.customr_num
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_non_personal_fy25 npers ON acc.customr_num = npers.customr_num AND acc.customr_bank_num = npers.customr_bank_num AND acc.customr_type = npers.customr_type
WHERE acc.customr_bank_num = 4
AND acc.customr_type = 1
AND acc.aplictn_id in ('ACS', 'VSA')
AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
--AND SUBSTRING(acc.ifw_effective_date, 1, 8) > '20241031' or SUBSTRING(acc.ifw_effective_date, 1, 8) is NULL
AND npers.customr_status = '00'
)
/*16784275*/

## Debug: Reconciliation of Metrics 1.1–1.5 vs 6.5b

**After NACO dedup:** The NACO table now deduplicates by `customr_num` to match 6.5b Method B.

**Current state (by `cust_no`):** Sum(1.1–1.5) = 16,784,682 vs 6.5b = 16,784,275 → **overlap of 407**

These debug cells investigate:
1. NULL `cust_id` records after CAEDW join
2. NACO deduplication verification
3. Full reconciliation summary
4. Missing customers not in any bucket
5. `cust_no` vs `cust_intrl_id` counting comparison
6. Bucket overlap detection

In [ ]:
# ------------------------------------------------------------------
# Debug Step 1: Check how many NACO records have NULL cust_intrl_id
# after the CAEDW join. These would be invisible to metrics 1.1-1.5
# since they all count by cust_intrl_id.
# ------------------------------------------------------------------
null_cust_id_count = niu_naco.filter(col('cust_id').isNull()).select('cust_no', 'cust_type').dropDuplicates().count()
total_niu_naco_count = niu_naco.select('cust_no', 'cust_type').dropDuplicates().count()
print(f"NACO records with NULL cust_id after CAEDW join: {null_cust_id_count}")
print(f"Total distinct NACO records (cust_no + cust_type): {total_niu_naco_count}")

In [ ]:
# ------------------------------------------------------------------
# Debug Step 2: Verify NACO deduplication
#   After the GROUP BY customr_num change, Method A and Method B
#   should now produce the same count. If dual_type_customers > 0,
#   it means the NACO table still has duplicates.
# ------------------------------------------------------------------
naco_by_num = naco.select('customr_num').distinct().count()
naco_by_num_type = naco.select('customr_num', 'customr_type').distinct().count()
dual_type_customers = naco_by_num_type - naco_by_num
print(f"Distinct customr_num (Method B / business): {naco_by_num}")
print(f"Distinct customr_num + customr_type (Method A): {naco_by_num_type}")
print(f"Customers appearing in BOTH personal & non-personal: {dual_type_customers}")

In [ ]:
# ------------------------------------------------------------------
# Debug Step 3: Compare the sum of metrics 1.1-1.5 with 6.5b
# Collect all metric counts and print a reconciliation summary.
# ------------------------------------------------------------------
m_1_1 = metric_1_1_count  # already computed above

m_1_2 = contact_center_hrc1.select('cust_intrl_id').distinct().count()
m_1_3 = contact_center_hrc.select('cust_intrl_id').distinct().count()
m_1_4 = contact_center_hrc2.select('cust_intrl_id').distinct().count()
m_1_5 = contact_center_hrc3.select('cust_intrl_id').distinct().count()

metric_sum = m_1_1 + m_1_2 + m_1_3 + m_1_4 + m_1_5

print("=== Reconciliation Summary ===")
print(f"  1.1 (Unscored):       {m_1_1:>12,}")
print(f"  1.2 (Tier 1/2):       {m_1_2:>12,}")
print(f"  1.3 (High):           {m_1_3:>12,}")
print(f"  1.4 (Medium):         {m_1_4:>12,}")
print(f"  1.5 (Low + unscored): {m_1_5:>12,}")
print(f"  ────────────────────────────")
print(f"  Sum(1.1–1.5):         {metric_sum:>12,}")
print(f"  6.5b (Method B):      {naco_by_num:>12,}")
print(f"  Variance:             {naco_by_num - metric_sum:>12,}")
print(f"  NULL cust_id records: {null_cust_id_count:>12,}")

In [ ]:
# ------------------------------------------------------------------
# Debug Step 4: Find NACO customers NOT captured by ANY of 1.1–1.5
# These are the "missing" customers that explain the variance.
# ------------------------------------------------------------------
from functools import reduce

# Collect all cust_intrl_ids from each metric bucket
all_metric_ids = reduce(
    lambda a, b: a.union(b),
    [
        metric_1_1.select(col('cust_no'), col('cust_type').alias('cust_type_mn')),
        contact_center_hrc1.select('cust_no', 'cust_type_mn'),
        contact_center_hrc.select('cust_no', 'cust_type_mn'),
        contact_center_hrc2.select('cust_no', 'cust_type_mn'),
        contact_center_hrc3.select('cust_no', 'cust_type_mn'),
    ]
).dropDuplicates()

# Left-anti join with niu_naco to find who's missing
missing_customers = niu_naco.join(all_metric_ids, on=['cust_no', 'cust_type_mn'], how='left_anti')
print(f"Customers in NACO but NOT in any metric bucket: {missing_customers.count()}")
display(missing_customers.select('cust_no', 'cust_type_mn', 'cust_id').limit(50))

In [ ]:
# ------------------------------------------------------------------
# Debug Step 6: Detect bucket overlap
#   Check if any cust_no appears in multiple metric buckets.
#   A negative Gap in Step 5 means some customers are double-counted
#   across buckets (e.g., appearing in both 1.4 and 1.5).
#   Latest run: 407 customers overlap, mostly [1.4, 1.5] = 381.
# ------------------------------------------------------------------
from pyspark.sql import DataFrame

# Tag each customer with their bucket(s)
bucket_1_1 = metric_1_1.select(col('cust_no')).distinct().withColumn('bucket', lit('1.1'))
bucket_1_2 = contact_center_hrc1.select('cust_no').distinct().withColumn('bucket', lit('1.2'))
bucket_1_3 = contact_center_hrc.select('cust_no').distinct().withColumn('bucket', lit('1.3'))
bucket_1_4 = contact_center_hrc2.select('cust_no').distinct().withColumn('bucket', lit('1.4'))
bucket_1_5 = contact_center_hrc3.select('cust_no').distinct().withColumn('bucket', lit('1.5'))

all_buckets = bucket_1_1.union(bucket_1_2).union(bucket_1_3).union(bucket_1_4).union(bucket_1_5)

# Find customers appearing in multiple buckets
from pyspark.sql.functions import collect_set, size
overlap = (all_buckets
    .groupBy('cust_no')
    .agg(collect_set('bucket').alias('buckets'), size(collect_set('bucket')).alias('num_buckets'))
    .filter(col('num_buckets') > 1)
)

overlap_count = overlap.count()
print(f"Customers in MULTIPLE buckets: {overlap_count:,}")
print()

# Show which bucket combinations have overlap
from pyspark.sql.functions import array_sort
overlap_summary = (overlap
    .withColumn('bucket_combo', array_sort('buckets'))
    .groupBy('bucket_combo')
    .count()
    .orderBy('count', ascending=False)
)
print("Overlap by bucket combination:")
display(overlap_summary)

# Count unique cust_no across ALL buckets (deduplicated)
unique_cust_no = all_buckets.select('cust_no').distinct().count()
print(f"\nUnique cust_no across all buckets (deduped): {unique_cust_no:,}")
print(f"6.5b (Method B):                              {naco_by_num:,}")
print(f"Remaining gap:                                {naco_by_num - unique_cust_no:,}")

In [ ]:
# ------------------------------------------------------------------
# Debug Step 6: Detect bucket overlap
# Check if any cust_no appears in multiple metric buckets.
# The 16,296 overshoot means some customers are double-counted.
# ------------------------------------------------------------------
from pyspark.sql import DataFrame

# Tag each customer with their bucket(s)
bucket_1_1 = metric_1_1.select(col('cust_no')).distinct().withColumn('bucket', lit('1.1'))
bucket_1_2 = contact_center_hrc1.select('cust_no').distinct().withColumn('bucket', lit('1.2'))
bucket_1_3 = contact_center_hrc.select('cust_no').distinct().withColumn('bucket', lit('1.3'))
bucket_1_4 = contact_center_hrc2.select('cust_no').distinct().withColumn('bucket', lit('1.4'))
bucket_1_5 = contact_center_hrc3.select('cust_no').distinct().withColumn('bucket', lit('1.5'))

all_buckets = bucket_1_1.union(bucket_1_2).union(bucket_1_3).union(bucket_1_4).union(bucket_1_5)

# Find customers appearing in multiple buckets
from pyspark.sql.functions import collect_set, size
overlap = (all_buckets
    .groupBy('cust_no')
    .agg(collect_set('bucket').alias('buckets'), size(collect_set('bucket')).alias('num_buckets'))
    .filter(col('num_buckets') > 1)
)

overlap_count = overlap.count()
print(f"Customers in MULTIPLE buckets: {overlap_count:,}")
print()

# Show which bucket combinations have overlap
from pyspark.sql.functions import array_sort
overlap_summary = (overlap
    .withColumn('bucket_combo', array_sort('buckets'))
    .groupBy('bucket_combo')
    .count()
    .orderBy('count', ascending=False)
)
print("Overlap by bucket combination:")
display(overlap_summary)

# Count unique cust_no across ALL buckets (deduplicated)
unique_cust_no = all_buckets.select('cust_no').distinct().count()
print(f"\nUnique cust_no across all buckets (deduped): {unique_cust_no:,}")
print(f"6.5b (Method B):                              {naco_by_num:,}")
print(f"Remaining gap:                                {naco_by_num - unique_cust_no:,}")

## Reconciliation: Findings & Recommended Next Steps

**After NACO deduplication (`GROUP BY customr_num`):**
- NACO table now matches 6.5b Method B (16,784,275 customers)
- Sum of 1.1–1.5 by `cust_no` = 16,784,682 → **407 over** due to bucket overlap
- Sum of 1.1–1.5 by `cust_intrl_id` = 16,671,566 → **112,709 under** due to many-to-one CIF mapping

**Remaining issues:**
1. **Bucket overlap (407 customers)** — same `cust_no` appears in 1.5 AND one of 1.2/1.3/1.4. These are customers with a rated record (Medium/High/Tier) AND an unrated record.
2. **cust_intrl_id undercounting (~113K)** — multiple `cust_no` values resolve to the same `cust_intrl_id` in CAEDW (account migrations, mergers).

**To make Sum(1.1–1.5) = 6.5b exactly:**
- Count metrics by `cust_no` instead of `cust_intrl_id`
- Apply priority-based bucket assignment to eliminate the 407 overlap
- Priority: 1.2 (Tier 1/2) > 1.3 (High) > 1.4 (Medium) > 1.5 (Low+unscored) > 1.1 (Unscored)

## SD2/ABAC — PEP (Politically Exposed Persons) 2025

Joins the PEP list with the NACO universe to identify how many
contact-centre customers are flagged as PEPs, broken down by PEP type.

In [ ]:
# ------------------------------------------------------------------
# SD2 PEP 2025
#   1. Load the exploded PEP list for 2025
#   2. Filter to CIF-prefixed entities
#   3. Derive cust_no and cust_type_mn
#   4. Inner join with NACO universe
#   5. Count distinct cust_intrl_id (total and non-null PEP_TYPE)
#   6. Summarize by PEP_TYPE
# ------------------------------------------------------------------

# SD2 PEP 2025
pep = spark.table("ra_adido_2025.pep_list_2025_exploded")
pep_cif = pep.filter(col('ENTITY').startswith('CIF'))
pep_cif = pep_cif.withColumn('cust_no', substring(col('ENTITY'), -9, 9)).withColumn('cust_type_mn', when(substring(col("ENTITY"), 8, 1) == "1", "N").otherwise("P"))
display(pep_cif)

naco_pep = pep_cif.join(niu_naco, on = ['cust_no', 'cust_type_mn'], how = 'inner')
naco_pep.agg(countDistinct('cust_intrl_id').alias('SD2')).display()
naco_pep.filter(col("PEP_TYPE").isNotNull()).agg(countDistinct('cust_intrl_id').alias('SD2')).display()

sd2_summary = naco_pep.groupBy("PEP_TYPE").count().orderBy("count", ascending=False)
sd2_summary_notNull = naco_pep.filter(col("PEP_TYPE").isNotNull()).groupBy("PEP_TYPE").count().orderBy("count", ascending=False)

display(sd2_summary)
display(sd2_summary_notNull)